# 07 – Tính độ đồng thuận gán nhãn giữa các thành viên

**Đầu vào** (`MyDrive/VNTrafficSign/output_dongthuan/`):
- `nhan_nop/nhan_<Tên>.zip`: bản export YOLO của từng thành viên (gán nhãn độc lập 150 ảnh của notebook 06)
- `KHONG_GUI/bang_ma.csv`, `KHONG_GUI/nhan_goc/`, `KHONG_GUI/data_goc.yaml`: từ notebook 06
- `goi_gan_nhan.zip`: để vẽ hình các ảnh bất đồng

**Cách đo** (mọi phép so là giữa HAI bộ nhãn trên cùng một ảnh):
1. **Ghép hộp**: ghép 1–1 tối ưu (thuật toán Hungary, `scipy.optimize.linear_sum_assignment`) theo IoU, không xét
   lớp; một cặp hộp được coi là cùng một đối tượng khi IoU ≥ `IOU_THR` (0,5 như tiêu chí PASCAL VOC / COCO AP50;
   báo thêm 0,75). Cách ghép này giống cách tính AP, nên kết quả so được với độ chính xác của mô hình.
2. **Đồng thuận phát hiện**: F1 = 2·M / (n_A + n_B), M = số cặp ghép được (đối xứng, không cần chọn ai là "đúng").
3. **Đồng thuận vị trí**: IoU trung bình của các cặp ghép được (Papadopoulos và cộng sự, ICCV 2017, dùng IoU giữa
   hai lần gán nhãn để đo độ nhất quán của hộp).
4. **Đồng thuận lớp** trên các cặp ghép được: tỉ lệ trùng lớp và **Cohen's kappa** (Cohen 1960) cho từng cặp người;
   **Fleiss' kappa** (Fleiss 1971) cho các đối tượng mà cả ba người cùng gán hộp. Diễn giải theo Landis & Koch (1977).
5. **F1 có xét lớp**: chỉ tính cặp ghép được VÀ cùng lớp.
6. Khoảng tin cậy 95%: bootstrap theo ảnh (lấy lại 150 ảnh có hoàn lại, `N_BOOT` lần).
Ngoài từng cặp thành viên, mỗi thành viên còn được so với **nhãn hiện có của bộ dữ liệu** (`Goc`) → ước lượng mức
khác biệt giữa nhãn của bộ dữ liệu và một lần gán nhãn độc lập.

**Đầu ra:** `output_dongthuan/ket_qua/`: `dong_thuan_tung_cap.csv`, `fleiss_kappa.json`, `bat_dong.csv`,
`hinh_bat_dong/*.png`, `tom_tat.json`

In [1]:
import io
import itertools
import json
import os
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/VNTrafficSign')
except ImportError:
    DRIVE_ROOT = Path(os.environ.get('VNTS_DRIVE_ROOT', './VNTrafficSign'))

OUT = DRIVE_ROOT / 'output_dongthuan'
SECRET = OUT / 'KHONG_GUI'
SUBMIT = OUT / 'nhan_nop'
RES = OUT / 'ket_qua'
(RES / 'hinh_bat_dong').mkdir(parents=True, exist_ok=True)

IOU_THRS = [0.5, 0.75]       # 0,5 là chỉ số chính
N_BOOT = 2000
SEED = 0
N_FIG = 24                    # số ảnh bất đồng vẽ ra để xem
# Ảnh mà người gán nhãn CỐ Ý để trống (không thấy biển nào): Roboflow không đưa ảnh chưa gán nhãn vào bản export, nên
# bản nộp sẽ thiếu ảnh đó. Ghi mã ảnh vào đây SAU KHI người đó xác nhận là cố ý để trống; ảnh được tính là "0 hộp"
# (một quyết định gán nhãn hợp lệ, được so như mọi ảnh khác). Ảnh thiếu mà không ghi ở đây → notebook dừng lại.
CO_Y_DE_TRONG = {
    'Tram': ['DT038'],
    'Anh': ['DT038'],
    'Dat': ['DT038']
}
try:
    display
except NameError:
    display = print

codes = pd.read_csv(SECRET / 'bang_ma.csv')['code'].tolist()
cfg = yaml.safe_load(open(SECRET / 'data_goc.yaml', encoding='utf-8'))
NAMES = list(cfg['names'].values()) if isinstance(cfg['names'], dict) else list(cfg['names'])
NAME2ID = {n: i for i, n in enumerate(NAMES)}
CODE_RE = re.compile(r'^(DT\d{3})(?:[_.].*)?$')

Mounted at /content/drive


## 1. Đọc nhãn: nhãn gốc + bản nộp của từng người (ánh xạ lớp theo TÊN, vì Roboflow có thể đổi thứ tự lớp)

In [2]:
def parse_lines(text, names_local):
    """YOLO → list (lớp_toàn_cục, x0, y0, x1, y1) chuẩn hóa; polygon → hộp bao ngoài (giống notebook 02–04)."""
    out = []
    for line in text.splitlines():
        v = line.split()
        if len(v) < 5:
            continue
        c, xs = int(float(v[0])), [float(t) for t in v[1:]]
        if len(xs) == 4:
            xc, yc, w, h = xs
            x0, y0, x1, y1 = xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2
        else:
            x0, y0, x1, y1 = min(xs[0::2]), min(xs[1::2]), max(xs[0::2]), max(xs[1::2])
        name = names_local[c]
        if name not in NAME2ID:
            raise SystemExit(f'❌ Tên lớp "{name}" không có trong classes.txt → người gán nhãn gõ sai tên lớp.')
        out.append((NAME2ID[name], max(0., x0), max(0., y0), min(1., x1), min(1., y1)))
    return out


labels = {'Goc': {c: parse_lines((SECRET / 'nhan_goc' / f'{c}.txt').read_text(encoding='utf-8'), NAMES) for c in codes}}
for z in sorted(SUBMIT.glob('nhan_*.zip')):
    who = z.stem[len('nhan_'):]
    with zipfile.ZipFile(z) as zf:
        # Zip export của Roboflow đôi khi chứa cùng một tên file 2 lần (ví dụ data.yaml). Chấp nhận nếu NỘI DUNG giống hệt;
        # khác nhau thì dừng, vì không biết bản nào đúng.
        infos = [i for i in zf.infolist() if not i.is_dir()]
        ymls = [i for i in infos if i.filename.endswith(('data.yaml', 'data.yml'))]
        if not ymls:
            raise SystemExit(f'❌ {z.name}: không có data.yaml → có thể export sai định dạng. Export lại định dạng YOLOv8. '
                             f'Một vài file trong zip: {[i.filename for i in infos[:8]]}')
        ynoi = {zf.read(i) for i in ymls}
        if len(ynoi) > 1:
            raise SystemExit(f'❌ {z.name}: có {len(ymls)} file data.yaml với nội dung KHÁC nhau: {[i.filename for i in ymls]} '
                             '→ có thể zip chứa 2 bản export. Export lại một bản duy nhất.')
        if len(ymls) > 1:
            print(f'ℹ️ {z.name}: {len(ymls)} bản data.yaml giống hệt nhau → dùng một bản.')
        y = yaml.safe_load(ynoi.pop())
        names_local = list(y['names'].values()) if isinstance(y['names'], dict) else list(y['names'])
        lab, raw, seen_img = {}, {}, set()
        for i in infos:
            p = Path(i.filename)
            m = CODE_RE.match(p.stem)
            if not m:
                continue
            if p.parent.name == 'images':
                seen_img.add(m.group(1))
            if p.parent.name == 'labels' and p.suffix == '.txt':
                txt = zf.read(i).decode('utf-8')
                if m.group(1) in raw:
                    if raw[m.group(1)].split() != txt.split():
                        raise SystemExit(f'❌ {z.name}: ảnh {m.group(1)} có 2 file nhãn với nội dung khác nhau '
                                         '(tải lên 2 lần và gán nhãn khác nhau?) → xóa bản thừa trên Roboflow rồi export lại.')
                    continue
                raw[m.group(1)] = txt
                lab[m.group(1)] = parse_lines(txt, names_local)
    thieu = sorted(set(codes) - seen_img - set(lab))
    chua_xac_nhan = [c for c in thieu if c not in CO_Y_DE_TRONG.get(who, [])]
    if chua_xac_nhan:
        raise SystemExit(f'❌ {z.name} thiếu {len(chua_xac_nhan)} ảnh: {chua_xac_nhan}. Hỏi {who}: nếu CỐ Ý để trống (không thấy '
                         f'biển) thì ghi vào CO_Y_DE_TRONG = {{"{who}": {chua_xac_nhan}}} ở ô cấu hình rồi chạy lại; '
                         f'nếu không, nộp lại bản export đủ {len(codes)} ảnh.')
    if thieu:
        print(f'ℹ️ {who}: {len(thieu)} ảnh cố ý để trống (tính là 0 hộp): {thieu}')
    labels[who] = {c: lab.get(c, []) for c in codes}          # có ảnh mà không có file nhãn = ảnh để trống
MEMBERS = [w for w in labels if w != 'Goc']
assert len(MEMBERS) >= 2, f'Cần ít nhất 2 bản nộp trong {SUBMIT}, thấy {MEMBERS}'
display(pd.DataFrame({w: {'so_hop': sum(len(v) for v in labels[w].values()),
                          'anh_trong': sum(not v for v in labels[w].values())} for w in labels}).T)

ℹ️ nhan_Anh.zip: 3 bản data.yaml giống hệt nhau → dùng một bản.
ℹ️ Anh: 1 ảnh cố ý để trống (tính là 0 hộp): ['DT038']
ℹ️ nhan_Dat.zip: 3 bản data.yaml giống hệt nhau → dùng một bản.
ℹ️ Dat: 1 ảnh cố ý để trống (tính là 0 hộp): ['DT038']
ℹ️ Tram: 1 ảnh cố ý để trống (tính là 0 hộp): ['DT038']


,so_hop,anh_trong
Goc,260,0
Anh,287,1
Dat,271,1
Tram,294,1


## 2. Ghép hộp và tính chỉ số cho từng cặp

In [3]:
def iou_matrix(A, B):
    if not A or not B:
        return np.zeros((len(A), len(B)))
    a, b = np.array([x[1:] for x in A]), np.array([x[1:] for x in B])
    ix0, iy0 = np.maximum(a[:, None, 0], b[None, :, 0]), np.maximum(a[:, None, 1], b[None, :, 1])
    ix1, iy1 = np.minimum(a[:, None, 2], b[None, :, 2]), np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix1 - ix0, 0, None) * np.clip(iy1 - iy0, 0, None)
    area = lambda t: (t[:, 2] - t[:, 0]) * (t[:, 3] - t[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-12)


def match(A, B, thr):
    """Ghép 1–1 tối ưu theo IoU (Hungary), giữ cặp IoU ≥ thr. Trả list (i, j, iou)."""
    M = iou_matrix(A, B)
    if M.size == 0:
        return []
    r, c = linear_sum_assignment(-M)
    return [(i, j, M[i, j]) for i, j in zip(r, c) if M[i, j] >= thr]


def cohen_kappa(y1, y2):
    if not y1:
        return float('nan')
    cats = sorted(set(y1) | set(y2))
    k = {c: i for i, c in enumerate(cats)}
    t = np.zeros((len(cats), len(cats)))
    for a, b in zip(y1, y2):
        t[k[a], k[b]] += 1
    n = t.sum(); po = np.trace(t) / n; pe = (t.sum(0) * t.sum(1)).sum() / n ** 2
    return float('nan') if pe == 1 else (po - pe) / (1 - pe)


def per_image_stats(A, B, thr):
    m = match(A, B, thr)
    return {'nA': len(A), 'nB': len(B), 'M': len(m), 'M_cung_lop': sum(A[i][0] == B[j][0] for i, j, _ in m),
            'ious': [u for _, _, u in m], 'ya': [A[i][0] for i, _, _ in m], 'yb': [B[j][0] for _, j, _ in m]}


def summarize(st):
    nA, nB, M = sum(s['nA'] for s in st), sum(s['nB'] for s in st), sum(s['M'] for s in st)
    Mc = sum(s['M_cung_lop'] for s in st)
    ious = [u for s in st for u in s['ious']]
    ya, yb = [c for s in st for c in s['ya']], [c for s in st for c in s['yb']]
    return {'n_hop_A': nA, 'n_hop_B': nB, 'n_ghep': M,
            'F1_phat_hien': 2 * M / (nA + nB) if nA + nB else float('nan'),
            'F1_co_lop': 2 * Mc / (nA + nB) if nA + nB else float('nan'),
            'IoU_tb': float(np.mean(ious)) if ious else float('nan'),
            'trung_lop_%': 100 * Mc / M if M else float('nan'),
            'cohen_kappa': cohen_kappa(ya, yb)}


rng = np.random.default_rng(SEED)
BOOT_IDX = [rng.integers(0, len(codes), len(codes)) for _ in range(N_BOOT)]
rows = []
for thr in IOU_THRS:
    for a, b in itertools.combinations(list(labels), 2):
        st = [per_image_stats(labels[a][c], labels[b][c], thr) for c in codes]
        row = {'IoU_nguong': thr, 'A': a, 'B': b, 'loai': 'Goc–thành viên' if 'Goc' in (a, b) else 'thành viên–thành viên',
               **summarize(st)}
        boots = [summarize([st[i] for i in idx]) for idx in BOOT_IDX]
        for k in ('F1_phat_hien', 'F1_co_lop', 'IoU_tb', 'cohen_kappa'):
            v = np.array([bt[k] for bt in boots], dtype=float); v = v[~np.isnan(v)]
            row[f'{k}_CI95'] = f'[{np.percentile(v, 2.5):.3f}, {np.percentile(v, 97.5):.3f}]' if len(v) else ''
        rows.append(row)
pair_df = pd.DataFrame(rows)
pair_df.to_csv(RES / 'dong_thuan_tung_cap.csv', index=False, encoding='utf-8-sig')
display(pair_df.round(4))

,IoU_nguong,A,B,loai,n_hop_A,n_hop_B,n_ghep,F1_phat_hien,F1_co_lop,IoU_tb,trung_lop_%,cohen_kappa,F1_phat_hien_CI95,F1_co_lop_CI95,IoU_tb_CI95,cohen_kappa_CI95
0,0.50,Goc,Anh,Goc–thành viên,260,287,247,0.9031,0.8739,0.8405,96.7611,0.9665,"[0.869, 0.933]","[0.836, 0.907]","[0.830, 0.851]","[0.941, 0.988]"
1,0.50,Goc,Dat,Goc–thành viên,260,271,234,0.8814,0.8588,0.8664,97.4359,0.9735,"[0.841, 0.916]","[0.816, 0.897]","[0.856, 0.877]","[0.950, 0.991]"
2,0.50,Goc,Tram,Goc–thành viên,260,294,249,0.8989,0.8881,0.8627,98.7952,0.9875,"[0.862, 0.932]","[0.849, 0.922]","[0.853, 0.872]","[0.971, 1.000]"
3,0.50,Anh,Dat,thành viên–thành viên,287,271,262,0.9391,0.8925,0.8587,95.0382,0.9489,"[0.918, 0.960]","[0.858, 0.925]","[0.849, 0.868]","[0.918, 0.975]"
4,0.50,Anh,Tram,thành viên–thành viên,287,294,276,0.9501,0.9225,0.8737,97.1014,0.9701,"[0.931, 0.968]","[0.897, 0.947]","[0.865, 0.882]","[0.946, 0.989]"
5,0.50,Dat,Tram,thành viên–thành viên,271,294,263,0.9310,0.9062,0.8876,97.3384,0.9726,"[0.908, 0.952]","[0.872, 0.937]","[0.879, 0.896]","[0.950, 0.992]"
6,0.75,Goc,Anh,Goc–thành viên,260,287,223,0.8154,0.7934,0.8579,97.3094,0.9721,"[0.764, 0.862]","[0.742, 0.843]","[0.851, 0.865]","[0.944, 0.995]"
7,0.75,Goc,Dat,Goc–thành viên,260,271,216,0.8136,0.7985,0.8819,98.1481,0.9809,"[0.766, 0.860]","[0.749, 0.847]","[0.875, 0.889]","[0.958, 0.995]"
8,0.75,Goc,Tram,Goc–thành viên,260,294,228,0.8231,0.8123,0.8789,98.6842,0.9864,"[0.775, 0.869]","[0.763, 0.859]","[0.873, 0.886]","[0.968, 1.000]"
9,0.75,Anh,Dat,thành viên–thành viên,287,271,246,0.8817,0.8351,0.8696,94.7154,0.9456,"[0.843, 0.917]","[0.789, 0.877]","[0.863, 0.876]","[0.913, 0.973]"


## 3. Fleiss' kappa cho các đối tượng mà mọi thành viên đều gán hộp (IoU ≥ 0,5 giữa MỌI cặp)

In [4]:
def landis_koch(k):
    if np.isnan(k):
        return ''
    for lim, txt in ((0, 'kém (poor)'), (0.2, 'nhẹ (slight)'), (0.4, 'khá (fair)'), (0.6, 'trung bình (moderate)'),
                     (0.8, 'đáng kể (substantial)'), (1.01, 'gần như hoàn toàn (almost perfect)')):
        if k <= lim:
            return txt
    return ''


def fleiss_kappa(items):
    """items: list các list nhãn lớp (mỗi đối tượng, mỗi người một nhãn, cùng số người)."""
    if not items:
        return float('nan')
    cats = sorted({c for it in items for c in it}); k = {c: i for i, c in enumerate(cats)}
    n = len(items[0]); t = np.zeros((len(items), len(cats)))
    for r, it in enumerate(items):
        for c in it:
            t[r, k[c]] += 1
    P = ((t * (t - 1)).sum(1) / (n * (n - 1))).mean(); pj = t.sum(0) / t.sum()
    Pe = (pj ** 2).sum()
    return float('nan') if Pe == 1 else (P - Pe) / (1 - Pe)


def full_agree_objects(code, who, thr=0.5):
    """Đối tượng mà mọi người trong `who` đều có hộp, ghép nhất quán (người đầu làm mốc, mọi cặp IoU ≥ thr)."""
    ref, others = who[0], who[1:]
    ms = {o: {i: j for i, j, _ in match(labels[ref][code], labels[o][code], thr)} for o in others}
    items = []
    for i, box in enumerate(labels[ref][code]):
        if not all(i in ms[o] for o in others):
            continue
        boxes = [box] + [labels[o][code][ms[o][i]] for o in others]
        if all(iou_matrix([p], [q])[0, 0] >= thr for p, q in itertools.combinations(boxes, 2)):
            items.append([b[0] for b in boxes])
    return items


items = [it for c in codes for it in full_agree_objects(c, MEMBERS)]
tot_ref = sum(len(labels[MEMBERS[0]][c]) for c in codes)
fk = fleiss_kappa(items)
boot_fk = []
per_code = {c: full_agree_objects(c, MEMBERS) for c in codes}
for idx in BOOT_IDX:
    v = fleiss_kappa([it for i in idx for it in per_code[codes[i]]])
    if not np.isnan(v):
        boot_fk.append(v)
fleiss = {'thanh_vien': MEMBERS, 'n_doi_tuong_moi_nguoi_deu_gan': len(items),
          'ti_le_tren_hop_cua_nguoi_moc_%': round(100 * len(items) / tot_ref, 2) if tot_ref else None,
          'fleiss_kappa': round(fk, 4), 'CI95': [round(float(np.percentile(boot_fk, q)), 4) for q in (2.5, 97.5)] if boot_fk else None,
          'dien_giai_Landis_Koch_1977': landis_koch(fk),
          'ti_le_doi_tuong_moi_nguoi_cung_lop_%': round(100 * np.mean([len(set(it)) == 1 for it in items]), 2) if items else None}
json.dump(fleiss, open(RES / 'fleiss_kappa.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
display(pd.Series(fleiss))

,0
thanh_vien,"[Anh, Dat, Tram]"
n_doi_tuong_moi_nguoi_deu_gan,257
ti_le_tren_hop_cua_nguoi_moc_%,89.55
fleiss_kappa,0.9653
CI95,"[0.9443, 0.9825]"
dien_giai_Landis_Koch_1977,gần như hoàn toàn (almost perfect)
ti_le_doi_tuong_moi_nguoi_cung_lop_%,94.94


## 4. Danh sách bất đồng (IoU 0,5) và hình minh họa

In [5]:
bd = []
for a, b in itertools.combinations(list(labels), 2):
    for c in codes:
        A, B = labels[a][c], labels[b][c]
        m = match(A, B, 0.5)
        mi, mj = {i for i, _, _ in m}, {j for _, j, _ in m}
        bd += [{'code': c, 'A': a, 'B': b, 'loai': f'chỉ {a} có hộp', 'lop_A': NAMES[A[i][0]], 'lop_B': ''}
               for i in range(len(A)) if i not in mi]
        bd += [{'code': c, 'A': a, 'B': b, 'loai': f'chỉ {b} có hộp', 'lop_A': '', 'lop_B': NAMES[B[j][0]]}
               for j in range(len(B)) if j not in mj]
        bd += [{'code': c, 'A': a, 'B': b, 'loai': 'khác lớp', 'lop_A': NAMES[A[i][0]], 'lop_B': NAMES[B[j][0]]}
               for i, j, _ in m if A[i][0] != B[j][0]]
bd_df = pd.DataFrame(bd, columns=['code', 'A', 'B', 'loai', 'lop_A', 'lop_B'])
bd_df.to_csv(RES / 'bat_dong.csv', index=False, encoding='utf-8-sig')
if len(bd_df):
    display(bd_df.groupby(['A', 'B', 'loai']).size().rename('so_luong').reset_index())
    kl = bd_df[bd_df['loai'] == 'khác lớp']
    if len(kl):
        display(kl.groupby(['lop_A', 'lop_B']).size().sort_values(ascending=False).head(20).rename('so_lan'))

COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#42d4f4']
pack = zipfile.ZipFile(OUT / 'goi_gan_nhan.zip')
img_names = {Path(n).stem: n for n in pack.namelist() if n.startswith('images/')}
mem_bd = bd_df[~((bd_df['A'] == 'Goc') | (bd_df['B'] == 'Goc'))]
for c in list(dict.fromkeys(mem_bd['code']))[:N_FIG]:
    im = Image.open(io.BytesIO(pack.read(img_names[c]))).convert('RGB')
    W, H = im.size; d = ImageDraw.Draw(im); lw = max(2, W // 400)
    for k, w in enumerate(list(labels)):
        for cl, x0, y0, x1, y1 in labels[w][c]:
            d.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline=COLORS[k % len(COLORS)], width=lw)
            d.text((x0 * W + 2, max(0, y0 * H - 12 - 12 * k)), f'{w}: {NAMES[cl]}', fill=COLORS[k % len(COLORS)])
    im.save(RES / 'hinh_bat_dong' / f'{c}.png')
print('Màu:', {w: COLORS[k % len(COLORS)] for k, w in enumerate(labels)})

tom = {'n_anh': len(codes), 'thanh_vien': MEMBERS, 'IoU_nguong_chinh': 0.5,
       'TB_F1_phat_hien_giua_thanh_vien': round(float(pair_df.query('IoU_nguong == 0.5 and loai == "thành viên–thành viên"')['F1_phat_hien'].mean()), 4),
       'TB_cohen_kappa_giua_thanh_vien': round(float(pair_df.query('IoU_nguong == 0.5 and loai == "thành viên–thành viên"')['cohen_kappa'].mean()), 4),
       'TB_F1_co_lop_thanh_vien_vs_Goc': round(float(pair_df.query('IoU_nguong == 0.5 and loai == "Goc–thành viên"')['F1_co_lop'].mean()), 4),
       'fleiss_kappa': fleiss['fleiss_kappa']}
json.dump(tom, open(RES / 'tom_tat.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
display(pd.Series(tom))
print('✅ Kết quả ở', RES)

,A,B,loai,so_luong
0,Anh,Dat,chỉ Anh có hộp,25
1,Anh,Dat,chỉ Dat có hộp,9
2,Anh,Dat,khác lớp,13
3,Anh,Tram,chỉ Anh có hộp,11
4,Anh,Tram,chỉ Tram có hộp,18
5,Anh,Tram,khác lớp,8
6,Dat,Tram,chỉ Dat có hộp,8
7,Dat,Tram,chỉ Tram có hộp,31
8,Dat,Tram,khác lớp,7
9,Goc,Anh,chỉ Anh có hộp,40


lop_A                                      lop_B                                    
123a-cam-re-trai                           123b-cam-re-phai                             4
306-toc-do-toi-thieu-cho-phep-50           127-toc-do-toi-da-cho-phep-50                4
123a-cam-re-trai                           103c-cam-oto-re-trai                         3
445g-bien-bao-kieu-mo-ta-tinh-trang-duong  244-doan-duong-hay-xay-ra-tai-nan            3
420-bat-dau-khu-dong-dan-cu                401-bat-dau-duong-uu-tien                    3
117-han-che-chieu-cao                      239b-chieu-cao-tinh-khong-thuc-te            3
438-ket-thuc-duong-cao-toc                 437-duong-cao-toc                            2
123b-cam-re-phai                           123a-cam-re-trai                             2
                                           103b-cam-oto-re-phai                         2
124a-cam-quay-dau-xe                       409-cho-quay-xe                              2
244-doan-duong-hay-xay-ra-tai-nan          445g-bien-bao-kieu-mo-ta-tinh-trang-duong    2
127-toc-do-toi-da-cho-phep-50              306-toc-do-toi-thieu-cho-phep-50             2
127-toc-do-toi-da-cho-phep-60              306-toc-do-toi-thieu-cho-phep-60             2
202b-nhieu-cho-ngoat-nguy-hiem-lien-tiep   201b-cho-ngoat-nguy-hiem                     2
245-di-cham                                239b-chieu-cao-tinh-khong-thuc-te            2
103b-cam-oto-re-phai                       123b-cam-re-phai                             1
127-toc-do-toi-da-cho-phep-40              127-toc-do-toi-da-cho-phep-60                1
137-cam-re-trai-va-re-phai                 123b-cam-re-phai                             1
239b-chieu-cao-tinh-khong-thuc-te          117-han-che-chieu-cao                        1
306-toc-do-toi-thieu-cho-phep-60           127-toc-do-toi-da-cho-phep-60                1
Name: so_lan, dtype: int64

Màu: {'Goc': '#e6194b', 'Anh': '#3cb44b', 'Dat': '#4363d8', 'Tram': '#f58231'}


,0
n_anh,150
thanh_vien,"[Anh, Dat, Tram]"
IoU_nguong_chinh,0.5
TB_F1_phat_hien_giua_thanh_vien,0.94
TB_cohen_kappa_giua_thanh_vien,0.9639
TB_F1_co_lop_thanh_vien_vs_Goc,0.8736
fleiss_kappa,0.9653


✅ Kết quả ở /content/drive/MyDrive/VNTrafficSign/output_dongthuan/ket_qua


In [6]:
# ===== Vẽ các ảnh cần xem lại: Goc | Anh | Dat | Tram =====
import io, zipfile
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

CAN_XEM = {
    'Bộ dữ liệu thiếu hộp mà cả 3 người đều khoanh': ['DT009', 'DT004', 'DT018', 'DT063', 'DT036', 'DT064', 'DT028',
                                                        'DT045', 'DT058', 'DT066', 'DT089', 'DT099', 'DT133'],
    'Bộ dữ liệu có hộp mà cả 3 người đều không khoanh': ['DT027', 'DT038', 'DT085', 'DT095', 'DT118'],
    'Cả 3 người gán khác lớp với bộ dữ liệu': ['DT067'],
}
WHO = ['Goc'] + [w for w in ['Anh', 'Dat', 'Tram'] if w in labels]
MAU = {'Goc': '#e6194b', 'Anh': '#3cb44b', 'Dat': '#4363d8', 'Tram': '#f58231'}
OUT_XEM = RES / 'hinh_can_xem'
OUT_XEM.mkdir(parents=True, exist_ok=True)

def ma_ngan(name):  # 127-toc-do-toi-da-cho-phep-60 -> 127-60 ; 102-cam-di-nguoc-chieu -> 102
    if name.startswith(('R.E-11a', 'R-E-11a')):
        return 'R.E-11a'
    p = name.split('-')
    return f'{p[0]}-{p[-1]}' if p[-1].isdigit() else p[0]

pack = zipfile.ZipFile(OUT / 'goi_gan_nhan.zip')
img_names = {Path(n).stem: n for n in pack.namelist() if n.startswith('images/')}

for nhom, ds in CAN_XEM.items():
    print(f'\n===== {nhom} =====')
    for c in ds:
        im = Image.open(io.BytesIO(pack.read(img_names[c]))).convert('RGB')
        W, H = im.size
        fig, axes = plt.subplots(1, len(WHO), figsize=(5 * len(WHO), 5 * H / W + 0.6))
        for ax, w in zip(axes, WHO):
            ax.imshow(im); ax.axis('off')
            boxes = labels[w][c]
            ax.set_title(f'{w}: {len(boxes)} hộp', fontsize=12, color=MAU[w])
            for cl, x0, y0, x1, y1 in boxes:
                ax.add_patch(patches.Rectangle((x0 * W, y0 * H), (x1 - x0) * W, (y1 - y0) * H,
                                               fill=False, edgecolor=MAU[w], linewidth=2))
                ax.text(x0 * W, max(0, y0 * H - 4), ma_ngan(NAMES[cl]), color='white', fontsize=8,
                        bbox=dict(facecolor=MAU[w], alpha=0.8, pad=1, edgecolor='none'))
        fig.suptitle(f'{c} – {nhom}', fontsize=12)
        fig.tight_layout()
        fig.savefig(OUT_XEM / f'{c}.png', dpi=110)
        plt.show()
        plt.close(fig)
print('Đã lưu vào', OUT_XEM)

Output hidden; open in https://colab.research.google.com to view.

In [7]:
# ===== Ước lượng tỉ lệ lỗi của nhãn bộ dữ liệu (Goc) so với nhãn đồng thuận của các thành viên =====
import itertools, json
from collections import Counter
import numpy as np

MEM = [w for w in labels if w != 'Goc']


def dong_thuan(code, k):
    """Đối tượng mà >= k thành viên cùng khoanh (IoU >= 0,5, nối theo chuỗi ghép từng cặp); lớp = lớp đa số."""
    nodes = [(w, i) for w in MEM for i in range(len(labels[w][code]))]
    idx = {n: t for t, n in enumerate(nodes)}
    parent = list(range(len(nodes)))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for a, b in itertools.combinations(MEM, 2):
        for i, j, _ in match(labels[a][code], labels[b][code], 0.5):
            parent[find(idx[(a, i)])] = find(idx[(b, j)])
    comp = {}
    for n in nodes:
        comp.setdefault(find(idx[n]), []).append(n)
    out = []
    for members in comp.values():
        nguoi = {w for w, _ in members}
        if len(nguoi) >= k:
            bs = [labels[w][code][i] for w, i in members]
            lop = Counter(b[0] for b in bs).most_common(1)[0][0]
            out.append((lop, *np.mean([b[1:] for b in bs], axis=0), len(nguoi)))
    return out


def dem(code, k):
    ref = dong_thuan(code, k)
    goc = labels['Goc'][code]
    m = match(ref, goc, 0.5)
    mr, mg = {i for i, _, _ in m}, {j for _, j, _ in m}
    thua = sum(1 for j in range(len(goc)) if j not in mg
               and not any(match([goc[j]], labels[w][code], 0.5) for w in MEM))
    sai_lop = sum(ref[i][0] != goc[j][0] for i, j, _ in m)
    return {'n_ref': len(ref), 'n_goc': len(goc), 'sot': len(ref) - len(mr), 'thua': thua, 'sai_lop': sai_lop, 'ghep': len(m)}


rng = np.random.default_rng(0)
ket_qua = {}
for k in (3, 2):
    rows = {c: dem(c, k) for c in codes}
    def ti_le(sub):
        S = {key: sum(rows[c][key] for c in sub) for key in ('n_ref', 'n_goc', 'sot', 'thua', 'sai_lop', 'ghep')}
        return {'sot_%': 100 * S['sot'] / S['n_ref'], 'thua_%': 100 * S['thua'] / S['n_goc'],
                'sai_lop_%': 100 * S['sai_lop'] / S['ghep']}, S
    tl, S = ti_le(codes)
    boot = [ti_le([codes[i] for i in rng.integers(0, len(codes), len(codes))])[0] for _ in range(2000)]
    ci = {key: [round(float(np.percentile([b[key] for b in boot], q)), 2) for q in (2.5, 97.5)] for key in tl}
    ket_qua[f'dong_thuan_toi_thieu_{k}_nguoi'] = {
        **S, **{key: round(v, 2) for key, v in tl.items()}, 'CI95': ci,
        'so_anh_co_bien_bi_sot': sum(rows[c]['sot'] > 0 for c in codes),
        'anh_co_bien_bi_sot': sorted(c for c in codes if rows[c]['sot'] > 0),
        'anh_co_hop_thua': sorted(c for c in codes if rows[c]['thua'] > 0),
        'anh_sai_lop': sorted(c for c in codes if rows[c]['sai_lop'] > 0)}
json.dump(ket_qua, open(RES / 'loi_nhan_goc.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
for k, v in ket_qua.items():
    print(f"\n{k}: tham chiếu {v['n_ref']} đối tượng, Goc {v['n_goc']} hộp")
    print(f"  biển bị sót  {v['sot']:3d}  = {v['sot_%']:.2f}%  CI95 {v['CI95']['sot_%']}  ({v['so_anh_co_bien_bi_sot']} ảnh)")
    print(f"  hộp thừa     {v['thua']:3d}  = {v['thua_%']:.2f}%  CI95 {v['CI95']['thua_%']}")
    print(f"  sai lớp      {v['sai_lop']:3d}  = {v['sai_lop_%']:.2f}%  CI95 {v['CI95']['sai_lop_%']}")
print('\nĐã lưu', RES / 'loi_nhan_goc.json')


dong_thuan_toi_thieu_3_nguoi: tham chiếu 257 đối tượng, Goc 260 hộp
  biển bị sót   26  = 10.12%  CI95 [4.94, 16.22]  (15 ảnh)
  hộp thừa       8  = 3.08%  CI95 [0.74, 6.25]
  sai lớp        2  = 0.87%  CI95 [0.0, 2.21]

dong_thuan_toi_thieu_2_nguoi: tham chiếu 287 đối tượng, Goc 260 hộp
  biển bị sót   39  = 13.59%  CI95 [7.91, 20.19]  (22 ảnh)
  hộp thừa       8  = 3.08%  CI95 [0.7, 6.08]
  sai lớp        3  = 1.21%  CI95 [0.0, 2.82]

Đã lưu /content/drive/MyDrive/VNTrafficSign/output_dongthuan/ket_qua/loi_nhan_goc.json
